# Titanic Data Preprocessing, Feature Engineering & Visualization

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read dataset (
df = pd.read_csv("titanic.csv")

Data Exploration

In [2]:

# === Preliminary Inspection ===
print("========== PRELIMINARY DATA INSPECTION ==========\n")
print(df.head(5))
print("\n=== Info ===")
print(df.info())
print("\n=== Descriptive statistics (all) ===")
print(df.describe().transpose())

========== PRELIMINARY DATA INSPECTION ==========

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4   

Data Cleaning and Descriptive Analysis

In [3]:

# Check missing values in 'Age'
print("Missing values before imputation:", df['Age'].isnull().sum())

# Calculate median Age per Pclass
median_per_class = df.groupby("Pclass")["Age"].median()
print("\nMedian Age per Pclass:\n", median_per_class)

# Impute missing Age values based on Pclass medianian
df['Age'] = df.apply( lambda row: median_per_class[row['Pclass']] if pd.isnull(row['Age']) else row['Age'], axis=1)
print("\nMissing values after imputation:", df['Age'].isnull().sum())

# Outlier Detection (Fare) dengan IQR
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print("\nNumber of potential outliers in Fare:", outliers.shape[0])


Missing values before imputation: 177

Median Age per Pclass:
 Pclass
1    37.0
2    29.0
3    24.0
Name: Age, dtype: float64

Missing values after imputation: 0

Number of potential outliers in Fare: 116


In [4]:

# Feature Engineering: Create 'FamilySize' feature
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
new_df = df[['Age', 'Fare', 'FamilySize']]
print("New 'FamilySize' feature created.")
print("First 5 rows of the new DataFrame:")
print(new_df.head(5))

# Create matrix for standardization and heartmap visualization
matrix = new_df.to_numpy()
print("\n Matrix shape:", end=" ")
print(matrix.shape)
print("\n Matrix head")
print(matrix[:5, :])

# Hitung mean (μ) dan std (σ) per kolom
mu = matrix.mean(axis=0)        # rata-rata tiap kolom
sigma = matrix.std(axis=0)      # std tiap kolom

# Standardisasi dengan broadcasting
Z = (matrix - mu) / sigma

# Verify
print("\nMeans (original):", mu)
print("Stds (original):", sigma)
print("Mean after scaling:", Z.mean(axis=0))
print("Stds after scaling:", Z.std(axis=0))

#save matrix
np.savetxt("matrix.csv", matrix, delimiter=",")

#save standardized data
np.savetxt("standarazied_data.csv", Z, delimiter=",")


New 'FamilySize' feature created.
First 5 rows of the new DataFrame:
    Age     Fare  FamilySize
0  22.0   7.2500           2
1  38.0  71.2833           2
2  26.0   7.9250           1
3  35.0  53.1000           2
4  35.0   8.0500           1

 Matrix shape: (891, 3)

 Matrix head
[[22.      7.25    2.    ]
 [38.     71.2833  2.    ]
 [26.      7.925   1.    ]
 [35.     53.1     2.    ]
 [35.      8.05    1.    ]]

Means (original): [29.06640853 32.20420797  1.90460157]
Stds (original): [13.23709736 49.66553444  1.61255287]
Mean after scaling: [ 2.19303314e-16  3.98733297e-18 -2.39239978e-17]
Stds after scaling: [1. 1. 1.]


In [5]:
# Feature Engineering: FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# ==========================
#  1. Violinplot
# ==========================
if all(c in df.columns for c in ['Sex', 'Survived', 'Age']):
    plt.figure(figsize=(8,6))
    # seaborn violinplot supports split only when hue has exactly 2 levels
    sns.violinplot(x='Sex', y='Age', hue='Survived', data=df, split=True, inner='quartile')
    plt.title("Age Distribution by Sex and Survival Status")
    plt.savefig(("violin_age_sex_survived.png"), dpi=300, bbox_inches='tight')
    plt.close()
else:
    print("Skipping violin: required columns missing")

# ==========================
#  2. Correlation Heatmap
# ==========================
corr = df[['Pclass', 'Age', 'Fare', 'FamilySize', 'Survived']].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap="coolwarm", cbar=True
)
plt.title("Correlation Heatmap (Numerical Features)")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300)
plt.close()

# ==========================
#  3. lmplot (Scatterplot with facets)
# ==========================
sns.lmplot(
    data=df,
    x="Age", y="Fare",
    hue="Survived", col="Pclass",
    fit_reg=False, 
    palette="Set1", height=5
)
plt.subplots_adjust(top=0.85)
plt.suptitle("Age vs Fare by Pclass, Colored by Survival", fontsize=14)
plt.savefig("lmplot_age_fare_by_pclass.png", dpi=300)
plt.close()

print("All plots saved: violinplot_age_sex_survived.png, correlation_heatmap.png, lmplot_age_fare_by_pclass.png")


All plots saved: violinplot_age_sex_survived.png, correlation_heatmap.png, lmplot_age_fare_by_pclass.png


In [6]:

# Save cleaned and engineered DataFrame (no index)
df.to_csv("titanic_cleaned_engineered.csv", index=False)
